# StateGen -- DS-1000 Experiments (Colab)

- **Runtime:** CPU only (no GPU needed)
- **Estimated time:** ~2-3 hrs for 200 tasks x 4 methods
- **Evaluation:** Real execution-based (not syntax check)
- **Dataset:** DS-1000 (100 Pandas + 100 Numpy = 200 tasks)

In [ ]:
%cd /content
!rm -rf StateGen
!git clone https://github.com/Joshh99/StateGen.git
%cd StateGen


In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install litellm python-dotenv datasets sentence-transformers

In [ ]:
from google.colab import files
uploaded = files.upload()
import shutil
shutil.move(list(uploaded.keys())[0], ".env")

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
key = os.getenv("TOGETHER_API_KEY")
print(f"TOGETHER_API_KEY: {'SET' if key else 'MISSING -- stop here'}")

## Configuration
Edit the variables below before running experiments.

In [ ]:
# Edit these as needed
PROVIDER = "together_ai"
MODEL = "deepseek-ai/DeepSeek-V3"
MAX_TASKS = 200        # 200 = full run (100 Pandas + 100 Numpy)
RESULTS_DIR = "results/ds1000"
# For ThetaEdge (future): set PROVIDER="openai_compatible" and
# add THETAEDGE_BASE_URL + THETAEDGE_API_KEY to your .env

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP = "/content/drive/MyDrive/StateGen_Results"
import os; os.makedirs(DRIVE_BACKUP, exist_ok=True)
print(f"Drive backup: {DRIVE_BACKUP}")

In [ ]:
!python experiments/run_ds1000.py \
  --method direct_gen \
  --max_tasks 2 \
  --dry_run

c:\Users\STUDENT\OneDrive\Documents\Projects\AI Research\StateGen
15:23:47 [INFO] __main__: Loading DS-1000 — max_tasks=2
15:23:48 [INFO] numexpr.utils: NumExpr defaulting to 12 threads.
15:23:48 [INFO] datasets: JAX version 0.9.0 available.
15:23:48 [INFO] data.ds1000_loader: Loading DS-1000 (split=test, libraries=['Pandas', 'Numpy'])
15:23:54 [INFO] data.ds1000_loader: Loaded 1 Pandas + 1 Numpy = 2 tasks
15:23:54 [INFO] __main__: Loaded 2 tasks.

DRY RUN — 2 tasks loaded, not calling LLM

-- Task 1: DS-1000/Pandas/0 (Pandas) --
Perturbation: Origin
Prompt (1114 chars):
Problem:
I have the following DataFrame:
    Col1  Col2  Col3  Type
0      1     2     3     1
1      4     5     6     1
2      7     8     9     2
3    10    11    12     2
4    13    14    15     3
5    16    17    18     3


The DataFrame is read from a CSV file. All rows which have Type 1 are on top, followed by the rows with Type 2, followed by the rows with Type 3, etc.
I would like to shuffle the order of the D

In [ ]:
# Run this first to sanity check before full 200-task run
import subprocess
result = subprocess.run([
    "python", "experiments/run_ds1000.py",
    "--method", "direct_gen",
    "--max_tasks", "10",
    "--provider", PROVIDER,
    "--model", MODEL,
    "--results_dir", RESULTS_DIR
], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

15:24:16 [INFO] __main__: Loading DS-1000 — max_tasks=10
15:24:17 [INFO] numexpr.utils: NumExpr defaulting to 12 threads.
15:24:18 [INFO] datasets: JAX version 0.9.0 available.
15:24:19 [INFO] data.ds1000_loader: Loading DS-1000 (split=test, libraries=['Pandas', 'Numpy'])
15:24:24 [INFO] data.ds1000_loader: Loaded 5 Pandas + 5 Numpy = 10 tasks
15:24:24 [INFO] __main__: Loaded 10 tasks.
15:24:24 [INFO] agents.llm_agent: LLMAgent ready: together_ai / deepseek-ai/DeepSeek-V3
15:24:24 [INFO] __main__: LLM: provider=together_ai, model=deepseek-ai/DeepSeek-V3
15:24:24 [INFO] __main__: 
Running method: direct_gen
15:24:24 [INFO] __main__: [direct_gen] Solving DS-1000/Pandas/0 ...

Provider List: https://docs.litellm.ai/docs/providers

15:24:24 [INFO] LiteLLM: 
LiteLLM completion() model= deepseek-ai/DeepSeek-V3; provider = deepseek

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

15:24:26 [

## Full Experiment Run
Check pilot results above before proceeding.
Expected cost: ~$3-5 on Together.ai for full 200 tasks x 4 methods.

In [ ]:
# Run sequentially to avoid rate limits
import subprocess
import shutil
METHODS = ["direct_gen", "self_planning", "self_debugging", "stategen"]
for method in METHODS:
    print(f"\n{'='*50}\nRunning: {method}\n{'='*50}")
    result = subprocess.run([
        "python", "experiments/run_ds1000.py",
        "--method", method,
        "--max_tasks", str(MAX_TASKS),
        "--provider", PROVIDER,
        "--model", MODEL,
        "--results_dir", RESULTS_DIR
    ], capture_output=True, text=True)
    print(result.stdout[-3000:])  # last 3000 chars to avoid cell overflow
    if result.returncode != 0:
        print("ERROR:", result.stderr[-1000:])
        break  # stop on first failure
    shutil.copytree(RESULTS_DIR, DRIVE_BACKUP, dirs_exist_ok=True)
    print(f"Backed up to Drive after {method}")

In [ ]:
import json, os
metrics_path = f"{RESULTS_DIR}/metrics.json"
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        m = json.load(f)
    print(json.dumps(m, indent=2))
else:
    print("No metrics file yet -- run experiments first")

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("ds1000_results", "zip", RESULTS_DIR)
files.download("ds1000_results.zip")